In [ ]:
import torch
import torch.nn as nn

In [ ]:
class GeLU(nn.Module):

    def __init__(self):
        super().__init__()
    def forward(self,x):

        return 0.5*x*(1+torch.tanh(torch.sqrt(torch.tensor(2.0/torch.pi))*(x+(0.044715*(x**3)))))

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self,d_in,d_out,num_heads):
        super().__init__()
        self.query_w=nn.Linear(d_in,d_out,bias=False)
        self.key_w=nn.Linear(d_in,d_out,bias=False)
        self.value_w=nn.Linear(d_in,d_out,bias=False)
        self.head_dim=d_out//num_heads

    
    def forward(self,X):
        batch,num_tokens,d_in=X.shape

        query=self.query_w(X)
        keys=self.key_w(X)
        value=self.value_w(X)


        keys=keys.view(batch,num_tokens,self.num_heads,self.head_dim)
        queries=query.view(batch,num_tokens,self.num_heads,self.head_dim)
        values=value.view(batch,num_tokens,self.num_heads,self.head_dim)
        
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores=torch.softmax(attn_scores/(keys.shape[-1]**0.5),dim=-1)
        context_vec = (attn_scores @ values).transpose(1, 2)

        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(batch, num_tokens, self.d_out)
 

        return context_vec


In [ ]:
class FeedForwardNetwork(nn.Module):
    def __init__(self,d_in):
        super().__init__()
        self.network=nn.Sequential(
            nn.Linear(d_in,d_in*4),
            nn.GeLU(),
            nn.Linear(d_in*4,d_in)
        )
    def forward(self,X):
        return self.network(X)


In [ ]:
class LayerNorm(nn.Module):
    def __init__(self,emb_dim):
        super().__init__()
        self.epies=1e-5
        self.shift=nn.Parameter(torch.zeros(emb_dim))
        self.scale=nn.Parameter(torch.ones(emb_dim))
    
    def forward(self,X):

        self.mean=X.mean(dim=-1,keepdim=True)
        self.variance=X.variance(-1,keepdim=True)
        norm_x=(X-self.mean)/((self.variance+self.epies)**0.5)

        return norm_x*self.scale+self.shift

In [ ]:
class EncoderTransformerBlock(nn.Module):
    def __init__(self,d_in,d_out,num_heads,dropout_rate):
        super().__init__()
        self.multiheadattention=MultiHeadAttention(d_in,d_out,num_heads,dropout_rate)
        self.feedforwardnetwork=FeedForwardNetwork(d_in)
        self.layerNorm1=LayerNorm(d_in)
        self.layerNorm2=LayerNorm(d_in)
        self.dropoutlayer=nn.Dropout(dropout_rate)

    def forward(self,X):
        
        shortcut=X
        X=self.layerNorm1(X)
        X=self.multiheadattention(X)
        X=self.dropoutlayer(X)
        X=X+shortcut
        shortcut=X
        X=self.layerNorm2(X)
        X=self.feedforwardnetwork(X)
        X=self.dropoutlayer(X)
        X=X+shortcut
        return X
        